# Topic: LAG and LEAD

## Definition (30-second explanation)
* `LAG()` and `LEAD()` are window functions that allow you to access values from previous (LAG) or subsequent (LEAD) rows within a result set, relative to the current row.
* They allow for cross-row comparisons (like calculating period-over-period growth) without requiring computationally expensive self-joins.

## Why Interviewers Ask This
* To test your ability to perform time-series analysis and calculate standard business metrics (MoM, YoY growth).
* To see if you can write clean, performant SQL (using window functions instead of nested subqueries/joins).
* To verify you know how to handle edge cases like `NULL` values on the first/last rows and division-by-zero errors.

## Core Concepts
* **Offset:** The second parameter in the function (e.g., `LAG(revenue, 1)`) determines how many rows to look back/forward. The default is 1.
* **Default Value:** The third parameter (e.g., `LAG(revenue, 1, 0)`) defines what to return if the target row doesn't exist (e.g., the very first row), preventing unexpected `NULL`s.
* **PARTITION BY:** Critical for resetting the window per group (e.g., looking at previous revenue *for the same product*, not just the absolute previous row).

## When to Use
* **LAG:** Calculating day-over-day or month-over-month changes, or finding the time elapsed between consecutive user events (sessionization).
* **LEAD:** Looking forward to predict next states, checking if a user upgraded in their next billing cycle, or identifying trend reversals.

## Advantages
* Dramatically simplifies SQL logic compared to self-joins.
* Highly performant since the database engine only needs to scan the data partition once.
* CTEs combined with LAG/LEAD make complex trend logic highly readable.

## Limitations
* Requires a strict, deterministic `ORDER BY` inside the `OVER()` clause to ensure adjacent rows are chronologically correct.
* The output is sensitive to missing data (e.g., if a month is missing from the table, `LAG(..., 1)` will grab the month before that, skipping a month).

## Common Comparisons
* **LAG vs LEAD:** `LAG` looks backwards (historical comparison), `LEAD` looks forwards (future outcome tracking). 
* **LAG vs Self-Join:** LAG is linear and clean; Self-Joins require joining a table on `t1.date = t2.date + 1`, which is slow and prone to duplication if relationships aren't 1:1.

## Common Interview Traps
* **Missing PARTITION BY:** If omitted, `LAG` might compare the January revenue of a 'Phone' to the December revenue of a 'Laptop' just because they are adjacent in the table.
* **Division by Zero:** When calculating growth percentage `(current - prev) / prev`, if `prev` is 0, the query crashes. Always use `NULLIF(prev, 0)`.
* **Cluttered SELECT clauses:** Writing the full `LAG(...)` window function multiple times in the same `SELECT` (once for absolute change, once for percentage). Use a CTE to define it once.

## Python / SQL Syntax
```sql
-- Standard syntax with offset (1) and default value (0)
LAG(column_name, 1, 0) OVER (
    PARTITION BY group_col 
    ORDER BY sort_col
)
```

## Important Formula
**Percentage Change:** (current_value - previous_value) / NULLIF(previous_value, 0)

## 45-Second Interview Answer
"LAG and LEAD are window functions used to access data from adjacent rows without needing self-joins. LAG looks backward, which is perfect for period-over-period growth or time-between-events, while LEAD looks forward. In interviews, the key to using them correctly is ensuring you have a strict ORDER BY for chronological alignment, a proper PARTITION BY so you don't accidentally compare different categories, and handling edge cases—like providing a default value for the first row to prevent NULLs, and using NULLIF to prevent division-by-zero when calculating growth rates."